# RiNALMo island-matching benchmark

Goal: find the best RiNALMo-based recipe to decide **which reference island matches which query island** (sparse bipartite matching on a shared structured core) and to localize that core.

**Ground truth is model-independent (never the RNA-FM pipeline output):**
- Rfam seed families: same-family pairs = true homologs; cross-family = true negatives.
- Synthetic divergence: controlled point-mutations + indels (homology and divergence known exactly).
- Planted core: a known structured core dropped into real intergenic flanks at a known offset (localization + island-length control).

Levers: representation (raw-1280 vs PCA k=16/32/64/128), score (whole-island MMD, sub-window argmin-MMD, nt-resolution SW on the cosine dotplot, Sinkhorn-OT), window strategy for long islands.

In [1]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import sys, json, gzip, random, math
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import torch

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modules.model_registry import load_model
from modules.pipeline.short_ncrna import _compute_mmd
from pyrion import TwoBitAccessor

SEED = 42
random.seed(SEED); np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
model, tokenize_fn, extract_fn = load_model('rinalmo', device)

_emb_cache = {}
def embed_once(seq):
    # per-token RiNALMo embedding (L, 1280), cached by sequence string
    if seq in _emb_cache:
        return _emb_cache[seq]
    rna = seq.upper().replace('T', 'U')
    tokens = tokenize_fn([rna])
    with torch.no_grad():
        reps = extract_fn(model, tokens)
    out = reps[0, 1:1+len(rna), :].cpu().float().numpy()
    _emb_cache[seq] = out
    return out

RFAM = REPO_ROOT / 'input_data/supply/rfam/Rfam.seed.gz'
HG38 = TwoBitAccessor(str(REPO_ROOT / 'input_data/2bit/hg38.2bit'))
print('device:', device, '| Rfam exists:', RFAM.exists())
_e = embed_once('ACGUACGUACGUACGUACGUACGUACGUACGU')
print('embed sanity:', _e.shape, _e.dtype)

device: mps | Rfam exists: True
embed sanity: (32, 1280) float32


In [2]:
from numpy.linalg import norm
from sklearn.decomposition import PCA

# ---------- representations ----------
def load_pca_npz(path):
    d = np.load(path)
    return {'mean': d['mean'].astype(np.float32), 'components': d['components'].astype(np.float32)}

PCA_MATCH = load_pca_npz(REPO_ROOT / 'modules/global_PCA/rinalmo_pca_k16.npz')     # k=16 (deployed matching)
PCA_FIND  = load_pca_npz(REPO_ROOT / 'modules/global_PCA/rinalmo_pca_find_k64.npz')  # k=64 (deployed finding)

def project(emb, pca):
    # emb: (L,1280) -> (L,k); pca=None keeps raw 1280
    if pca is None:
        return emb
    return (emb - pca['mean']) @ pca['components'].T

def fit_pca(pool, k):
    p = PCA(n_components=k, random_state=SEED).fit(pool)
    return {'mean': p.mean_.astype(np.float32), 'components': p.components_.astype(np.float32)}

# ---------- scores ----------
# MMD: LOWER = more similar. SW/cosine: HIGHER = more similar.
def whole_mmd(A, B):
    return _compute_mmd(A, B)

def best_window_mmd(A, B, win=128, stride=16):
    la, lb = len(A), len(B)
    if la < win or lb < win:
        return _compute_mmd(A, B)
    best = np.inf
    for i in range(0, la - win + 1, stride):
        Ai = A[i:i+win]
        for j in range(0, lb - win + 1, stride):
            m = _compute_mmd(Ai, B[j:j+win])
            if m < best:
                best = m
    return best

def cosine_dotplot(A, B):
    An = A / (norm(A, axis=1, keepdims=True) + 1e-8)
    Bn = B / (norm(B, axis=1, keepdims=True) + 1e-8)
    return An @ Bn.T

def sw_dotplot_score(A, B, tau=0.5, gap=0.3):
    # Smith-Waterman local alignment on the per-token cosine dotplot.
    # Returns (best_local_score, (i_end,j_end)). Higher = stronger conserved core.
    S = cosine_dotplot(A, B) - tau
    la, lb = S.shape
    H = np.zeros((la + 1, lb + 1), dtype=np.float32)
    best = 0.0; bi = bj = 0
    for i in range(1, la + 1):
        Srow = S[i - 1]; Hprev = H[i - 1]; Hi = H[i]
        for j in range(1, lb + 1):
            v = Hprev[j - 1] + Srow[j - 1]
            u = Hprev[j] - gap
            l = Hi[j - 1] - gap
            if u > v: v = u
            if l > v: v = l
            if v < 0.0: v = 0.0
            Hi[j] = v
            if v > best:
                best = v; bi = i; bj = j
    return float(best), (bi, bj)

print('representations: raw-1280, PCA_MATCH k=%d, PCA_FIND k=%d' % (PCA_MATCH['components'].shape[0], PCA_FIND['components'].shape[0]))
# quick sanity on the sample embedding
_A = project(_e, PCA_FIND); _B = project(_e, PCA_FIND)
print('self whole_mmd (should be ~0):', round(whole_mmd(_A, _B), 5))
print('self sw_dotplot best (raw):', round(sw_dotplot_score(_e, _e)[0], 3))

representations: raw-1280, PCA_MATCH k=16, PCA_FIND k=64
self whole_mmd (should be ~0): 0.0
self sw_dotplot best (raw): 16.0


## 1. Ground truth (model-independent)

**Rfam same-family pairs = true homologs** (share curated consensus structure); **cross-family pairs = true negatives**. This replaces the old 'agree with the RNA-FM pipeline' target.

In [3]:
def parse_rfam_seed(path):
    fams = {}
    acc = idd = None
    seqs = defaultdict(str)
    with gzip.open(path, 'rt', encoding='latin-1') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('#=GF AC'):
                acc = line.split()[2]
            elif line.startswith('#=GF ID'):
                idd = line.split()[2]
            elif line == '//':
                if acc:
                    ung = []
                    for s in seqs.values():
                        u = ''.join(s.upper().replace('T','U').replace('.','').replace('-','').split())
                        if u and set(u) <= set('ACGU'):
                            ung.append(u)
                    if ung:
                        fams[acc] = {'id': idd, 'seqs': ung}
                acc = idd = None
                seqs = defaultdict(str)
            elif line and not line.startswith('#'):
                parts = line.split(None, 1)
                if len(parts) == 2:
                    seqs[parts[0]] += parts[1]
    return fams

MIN_MEMBERS = 8
LEN_LO, LEN_HI = 50, 300
N_FAMS = 120
PAIRS_PER_FAM = 4
rng = random.Random(SEED)

fams = parse_rfam_seed(RFAM)
qual = []
for acc, v in fams.items():
    ss = [s for s in v['seqs'] if LEN_LO <= len(s) <= LEN_HI]
    if len(ss) >= MIN_MEMBERS:
        qual.append((acc, ss))
rng.shuffle(qual)
qual = qual[:N_FAMS]
fam_seqs = {acc: ss for acc, ss in qual}
accs = list(fam_seqs.keys())

pos_pairs = []
for acc, ss in qual:
    ss = list(ss); rng.shuffle(ss)
    for k in range(min(PAIRS_PER_FAM, len(ss) // 2)):
        pos_pairs.append((ss[2*k], ss[2*k+1], acc, acc))

neg_pairs = []
for _ in range(len(pos_pairs)):
    fa, fb = rng.sample(accs, 2)
    neg_pairs.append((rng.choice(fam_seqs[fa]), rng.choice(fam_seqs[fb]), fa, fb))

print('parsed families:', len(fams), '| qualifying (>=%d members, %d-%dnt):' % (MIN_MEMBERS, LEN_LO, LEN_HI), len(qual))
print('positive pairs (same family):', len(pos_pairs), '| negative pairs (cross family):', len(neg_pairs))
print('example pos families:', [p[2] for p in pos_pairs[:5]])
_pl = [len(p[0]) for p in pos_pairs]
print('pair seq length p10/50/90:', np.percentile(_pl, [10,50,90]).astype(int).tolist())

parsed families: 4227 | qualifying (>=8 members, 50-300nt): 120
positive pairs (same family): 480 | negative pairs (cross family): 480
example pos families: ['RF03530', 'RF03530', 'RF03530', 'RF03530', 'RF00211']
pair seq length p10/50/90: [64, 86, 167]


In [4]:
# Synthetic divergence + planted-core helpers (exact homology & core coordinates known).
_BASES = 'ACGU'

def mutate(seq, sub_rate=0.0, indel_rate=0.0, rng=None):
    rng = rng or random.Random(0)
    out = []
    for ch in seq:
        r = rng.random()
        if r < indel_rate / 2:
            continue                      # deletion
        if r < indel_rate:
            out.append(rng.choice(_BASES))  # insertion before ch
        if rng.random() < sub_rate:
            out.append(rng.choice([b for b in _BASES if b != ch]))
        else:
            out.append(ch)
    return ''.join(out)

_CHROMS = [c for c in HG38.chrom_sizes() if c.startswith('chr') and '_' not in c and c != 'chrM']
def random_intergenic(length, rng):
    cs = HG38.chrom_sizes()
    for _ in range(200):
        c = rng.choice(_CHROMS); s = rng.randint(0, cs[c] - length)
        q = str(HG38.fetch(c, s, s + length)).upper().replace('T', 'U')
        if 'N' not in q and len(q) == length:
            return q
    return 'A' * length

def planted_pair(core, flank_len, sub_rate=0.0, indel_rate=0.0, rng=None):
    # core dropped into DIFFERENT random flanks in ref and query, query core mutated.
    rng = rng or random.Random(0)
    lf, rf = random_intergenic(flank_len, rng), random_intergenic(flank_len, rng)
    ref = lf + core + rf
    ref_span = (len(lf), len(lf) + len(core))
    cq = mutate(core, sub_rate, indel_rate, rng)
    lf2, rf2 = random_intergenic(flank_len, rng), random_intergenic(flank_len, rng)
    q = lf2 + cq + rf2
    q_span = (len(lf2), len(lf2) + len(cq))
    return ref, ref_span, q, q_span

# demo
_rng = random.Random(1)
_core = fam_seqs[accs[0]][0]
_d10 = mutate(_core, sub_rate=0.10, rng=_rng)
print('mutate demo: core len %d -> 10%% sub len %d' % (len(_core), len(_d10)))
_r, _rs, _q, _qs = planted_pair(_core, flank_len=120, sub_rate=0.10, rng=_rng)
print('planted: ref len %d core@%s | query len %d core@%s' % (len(_r), _rs, len(_q), _qs))

mutate demo: core len 123 -> 10% sub len 123
planted: ref len 363 core@(120, 243) | query len 363 core@(120, 243)


## 2. Exp A - representation sweep

Does matching discrimination improve with PCA dimensionality (the k=16->64 lesson from finding)? Score = whole-island MMD; metric = AUC separating same-family (homolog) from cross-family pairs. Nested PCA fit on the same token pool isolates the dimensionality effect.

In [5]:
from sklearn.metrics import roc_auc_score

# embed every unique pair sequence once (cached), and build a token pool for nested PCA
uniq = set()
for a, b, _, _ in pos_pairs + neg_pairs:
    uniq.add(a); uniq.add(b)
uniq = list(uniq)
print('unique sequences to embed:', len(uniq))
pool = []
for i, s in enumerate(uniq):
    emb = embed_once(s)
    idx = np.random.RandomState(i).choice(len(emb), min(len(emb), 40), replace=False)
    pool.append(emb[idx])
    if (i + 1) % 250 == 0:
        print('  embedded', i + 1, '/', len(uniq))
pool = np.vstack(pool).astype(np.float32)
print('token pool for PCA:', pool.shape)

pca128 = fit_pca(pool, 128)
def sliced(k):
    return {'mean': pca128['mean'], 'components': pca128['components'][:k]}

reps = {
    'raw-1280': None,
    'deployed match k16': PCA_MATCH,
    'deployed find k64': PCA_FIND,
    'fit k16': sliced(16),
    'fit k32': sliced(32),
    'fit k64': sliced(64),
    'fit k128': sliced(128),
}

def auc_for(rep):
    labels = []; scores = []
    for grp, prs in [(1, pos_pairs), (0, neg_pairs)]:
        for a, b, _, _ in prs:
            A = project(embed_once(a), rep); B = project(embed_once(b), rep)
            scores.append(whole_mmd(A, B)); labels.append(grp)
    return roc_auc_score(labels, -np.array(scores))

print('\nExp A: whole-island MMD, same-family vs cross-family AUC')
print('%-22s %8s' % ('representation', 'AUC'))
expA = {}
for name, rep in reps.items():
    expA[name] = auc_for(rep)
    print('%-22s %8.3f' % (name, expA[name]))

unique sequences to embed: 1414
  embedded 250 / 1414
  embedded 500 / 1414
  embedded 750 / 1414
  embedded 1000 / 1414
  embedded 1250 / 1414
token pool for PCA: (56560, 1280)

Exp A: whole-island MMD, same-family vs cross-family AUC
representation              AUC
raw-1280                  0.941
deployed match k16        0.904
deployed find k64         0.700
fit k16                   0.920
fit k32                   0.931
fit k64                   0.940
fit k128                  0.943


## 3. Exp B - score functions in the flank-diluted regime

The realistic case: a conserved core sits inside **unrelated flanks**. Positives = same core (10%-diverged) planted in different random-intergenic flanks; negatives = **different-family cores** each in their own flanks (both islands are structured, but not the same structure). Whole-island MMD should dilute; sub-window / SW / OT should recover the core. Metric: match/non-match AUC.

In [6]:
rngB = random.Random(SEED + 7)
FLANK_B = 100
SUB_B = 0.10
N_B = 140

planted_pos = []   # (ref, ref_core_span, query, query_core_span)  same core, diverged
planted_neg = []   # different-family cores, each in its own flanks
for _ in range(N_B):
    fam = rngB.choice(accs)
    core = rngB.choice(fam_seqs[fam])
    planted_pos.append(planted_pair(core, FLANK_B, sub_rate=SUB_B, rng=rngB))
for _ in range(N_B):
    fa, fb = rngB.sample(accs, 2)
    coreA = rngB.choice(fam_seqs[fa]); coreB = rngB.choice(fam_seqs[fb])
    lf, rf = random_intergenic(FLANK_B, rngB), random_intergenic(FLANK_B, rngB)
    ref = lf + coreA + rf; rs = (len(lf), len(lf) + len(coreA))
    lf2, rf2 = random_intergenic(FLANK_B, rngB), random_intergenic(FLANK_B, rngB)
    q = lf2 + coreB + rf2; qs = (len(lf2), len(lf2) + len(coreB))
    planted_neg.append((ref, rs, q, qs))

# embed all planted islands once (cached)
n_emb = 0
for grp in (planted_pos, planted_neg):
    for ref, rs, q, qs in grp:
        embed_once(ref); embed_once(q); n_emb += 2
print('planted positives:', len(planted_pos), '| negatives:', len(planted_neg), '| islands embedded:', n_emb)
_ex = planted_pos[0]
print('example island lengths: ref %d (core %s), query %d (core %s)' % (len(_ex[0]), _ex[1], len(_ex[2]), _ex[3]))

planted positives: 140 | negatives: 140 | islands embedded: 560
example island lengths: ref 271 (core (100, 171)), query 271 (core (100, 171))


In [7]:
def sinkhorn_dist(A, B, eps=0.1, iters=30):
    C = (A * A).sum(1)[:, None] + (B * B).sum(1)[None, :] - 2.0 * (A @ B.T)
    C = np.maximum(C, 0.0); C = C / (C.mean() + 1e-9)
    K = np.exp(-C / eps)
    r = np.ones(len(A)); c = np.ones(len(B))
    u = np.full(len(A), 1.0 / len(A)); v = np.full(len(B), 1.0 / len(B))
    for _ in range(iters):
        r = u / (K @ c + 1e-9); c = v / (K.T @ r + 1e-9)
    Pm = r[:, None] * K * c[None, :]
    return float((Pm * C).sum())

REP = sliced(64)   # best cheap representation from Exp A
def Pj(seq):
    return project(embed_once(seq), REP)

scores = {'whole_mmd': [], 'window_mmd': [], 'sw_dotplot': [], 'sinkhorn': [], 'core_oracle_mmd': []}
labels = []
for grp, data in [(1, planted_pos), (0, planted_neg)]:
    for ref, rs, q, qs in data:
        A = Pj(ref); B = Pj(q)
        scores['whole_mmd'].append(whole_mmd(A, B))
        scores['window_mmd'].append(best_window_mmd(A, B, win=64, stride=16))
        scores['sw_dotplot'].append(sw_dotplot_score(A, B)[0])
        scores['sinkhorn'].append(sinkhorn_dist(A, B))
        scores['core_oracle_mmd'].append(whole_mmd(A[rs[0]:rs[1]], B[qs[0]:qs[1]]))
        labels.append(grp)
    print('scored', 'positives' if grp == 1 else 'negatives')
labels = np.array(labels)

higher_better = {'sw_dotplot'}
print('\nExp B: match/non-match AUC (shared core in unrelated flanks), rep=k64')
print('%-18s %8s' % ('score', 'AUC'))
expB = {}
for name, sc in scores.items():
    sc = np.array(sc, dtype=float)
    s = sc if name in higher_better else -sc
    expB[name] = roc_auc_score(labels, s)
    tag = ' (oracle: flanks removed)' if name == 'core_oracle_mmd' else ''
    print('%-18s %8.3f%s' % (name, expB[name], tag))

scored positives
scored negatives

Exp B: match/non-match AUC (shared core in unrelated flanks), rep=k64
score                   AUC
whole_mmd             0.617
window_mmd            0.651
sw_dotplot            0.993
sinkhorn              0.695
core_oracle_mmd       0.926 (oracle: flanks removed)


### Cost: is SW-on-dotplot harder than window-MMD?

Key question. Old RNA-FM cost per gene = A*B island pairs, and per pair a window-MMD matrix of ~(L/stride)^2 cells, each cell an MMD over window^2 token kernels -> the CPU disaster. The dotplot computes each token-token similarity ONCE (a single BLAS matmul); SW is then O(L_a*L_b) scalar ops. So per-pair the redundant (window/stride)^2 blowup disappears. Embedding is embed-once per island (A+B passes), not per window. Measured below (pure-Python SW is a prototype artifact; compiled SW is ~50-100x faster).

In [8]:
import time

sample = list(planted_pos + planted_neg)
random.Random(0).shuffle(sample)
sample = sample[:40]
# warm projection cache
for ref, rs, q, qs in sample:
    Pj(ref); Pj(q)

def per_pair_time(fn, reps=2):
    t0 = time.time()
    for _ in range(reps):
        for ref, rs, q, qs in sample:
            fn(Pj(ref), Pj(q))
    return (time.time() - t0) / (reps * len(sample))

def n_window_cells(A, B, win=64, stride=16):
    la, lb = len(A), len(B)
    if la < win or lb < win:
        return 1
    return len(range(0, la - win + 1, stride)) * len(range(0, lb - win + 1, stride))

avg_L = np.mean([(len(Pj(r)) + len(Pj(q))) / 2 for r, _, q, _ in sample])
avg_cells = np.mean([n_window_cells(Pj(r), Pj(q)) for r, _, q, _ in sample])

t_whole = per_pair_time(lambda A, B: whole_mmd(A, B))
t_win = per_pair_time(lambda A, B: best_window_mmd(A, B, win=64, stride=16))
t_dot = per_pair_time(lambda A, B: cosine_dotplot(A, B))
t_sw = per_pair_time(lambda A, B: sw_dotplot_score(A, B))

print('avg island length: %d nt   (rep=k64)' % round(avg_L))
print('per-pair CPU cost:')
print('  window_mmd 64/16  : %7.2f ms   (%.0f MMD cells/pair, current-style)' % (t_win * 1e3, avg_cells))
print('  whole_mmd         : %7.2f ms' % (t_whole * 1e3))
print('  cosine dotplot    : %7.3f ms   (one BLAS matmul, computed once)' % (t_dot * 1e3))
print('  sw_dotplot (py)   : %7.2f ms   (pure-Python DP prototype)' % (t_sw * 1e3))
print('  sw_dotplot (est. compiled ~70x) : %7.3f ms' % (t_sw * 1e3 / 70))
print()
print('AUC recap: window_mmd %.3f  vs  sw_dotplot %.3f' % (expB['window_mmd'], expB['sw_dotplot']))
print('=> SW-on-dotplot is both more ACCURATE and, compiled, cheaper per pair than window-MMD.')

avg island length: 305 nt   (rep=k64)
per-pair CPU cost:
  window_mmd 64/16  :   64.93 ms   (249 MMD cells/pair, current-style)
  whole_mmd         :    4.05 ms
  cosine dotplot    :   0.297 ms   (one BLAS matmul, computed once)
  sw_dotplot (py)   :   39.36 ms   (pure-Python DP prototype)
  sw_dotplot (est. compiled ~70x) :   0.562 ms

AUC recap: window_mmd 0.651  vs  sw_dotplot 0.993
=> SW-on-dotplot is both more ACCURATE and, compiled, cheaper per pair than window-MMD.


## 4. Exp C - localization correctness + long-island cost

Does the SW-aligned band land on the TRUE planted core (so emitted coordinates are trustworthy)? And does O(L^2) stay affordable for long islands?

In [9]:
def sw_dotplot_align(A, B, tau=0.5, gap=0.3):
    # SW with traceback -> (score, ref_range, query_range) at nt resolution
    S = cosine_dotplot(A, B) - tau
    la, lb = S.shape
    H = np.zeros((la + 1, lb + 1), dtype=np.float32)
    ptr = np.zeros((la + 1, lb + 1), dtype=np.int8)
    best = 0.0; bi = bj = 0
    for i in range(1, la + 1):
        Srow = S[i - 1]; Hp = H[i - 1]; Hi = H[i]; Pi = ptr[i]
        for j in range(1, lb + 1):
            d = Hp[j - 1] + Srow[j - 1]; u = Hp[j] - gap; l = Hi[j - 1] - gap
            v = 0.0; p = 0
            if d > v: v = d; p = 1
            if u > v: v = u; p = 2
            if l > v: v = l; p = 3
            Hi[j] = v; Pi[j] = p
            if v > best: best = v; bi = i; bj = j
    i, j = bi, bj; ri = []; qj = []
    while i > 0 and j > 0 and H[i, j] > 0:
        p = ptr[i, j]
        if p == 1: ri.append(i - 1); qj.append(j - 1); i -= 1; j -= 1
        elif p == 2: i -= 1
        elif p == 3: j -= 1
        else: break
    if not ri:
        return best, (0, 0), (0, 0)
    return best, (min(ri), max(ri) + 1), (min(qj), max(qj) + 1)

def ov_frac(rng_pred, span):
    inter = max(0, min(rng_pred[1], span[1]) - max(rng_pred[0], span[0]))
    return inter / max(1, span[1] - span[0])

loc_ref, loc_q, both = [], [], 0
for ref, rs, q, qs in planted_pos:
    _, rr, qr = sw_dotplot_align(Pj(ref), Pj(q))
    fr = ov_frac(rr, rs); fq = ov_frac(qr, qs)
    loc_ref.append(fr); loc_q.append(fq)
    if fr >= 0.5 and fq >= 0.5:
        both += 1
n = len(planted_pos)
print('Localization vs TRUE core (planted positives, n=%d):' % n)
print('  aligned band covers >=50%% of true core in BOTH ref & query: %d/%d (%.0f%%)' % (both, n, 100*both/n))
print('  median core-overlap: ref %.2f, query %.2f' % (np.median(loc_ref), np.median(loc_q)))

# long-island cost point
_core = fam_seqs[accs[0]][0]
_r, _rs, _q, _qs = planted_pair(_core, flank_len=400, sub_rate=0.10, rng=random.Random(3))
A, B = Pj(_r), Pj(_q)
t0 = time.time(); sc, rr, qr = sw_dotplot_align(A, B); dt = time.time() - t0
print('\nLong island: len ref %d / query %d (core@%s)' % (len(_r), len(_q), _rs))
print('  SW-align (pure py) %.0f ms -> compiled ~%.1f ms; band ref %s q %s; core-overlap ref %.2f' % (dt*1e3, dt*1e3/70, rr, qr, ov_frac(rr, _rs)))

Localization vs TRUE core (planted positives, n=140):
  aligned band covers >=50% of true core in BOTH ref & query: 140/140 (100%)
  median core-overlap: ref 1.00, query 1.00

Long island: len ref 923 / query 923 (core@(400, 523))
  SW-align (pure py) 413 ms -> compiled ~5.9 ms; band ref (316, 524) q (309, 524); core-overlap ref 1.00


In [10]:
# Which representation for SW-dotplot? Test on the hard flank-diluted regime (planted pos/neg).
sw_reps = {'deployed match k16': PCA_MATCH, 'fit k32': sliced(32), 'fit k64': sliced(64), 'raw-1280': None}
print('SW-dotplot match/non-match AUC by representation (flank-diluted):')
for name, rep in sw_reps.items():
    labs, scs = [], []
    for grp, data in [(1, planted_pos), (0, planted_neg)]:
        for ref, rs, q, qs in data:
            scs.append(sw_dotplot_score(project(embed_once(ref), rep), project(embed_once(q), rep))[0])
            labs.append(grp)
    print('  %-20s AUC %.3f' % (name, roc_auc_score(labs, scs)))

SW-dotplot match/non-match AUC by representation (flank-diluted):
  deployed match k16   AUC 0.993
  fit k32              AUC 0.995
  fit k64              AUC 0.993
  raw-1280             AUC 0.994


## Verdict

**Recipe for the RiNALMo matcher: embed each island once -> per-token cosine dotplot (deployed k16 matching PCA) -> nt-resolution Smith-Waterman (traceback -> aligned core coords + score).**

| finding | evidence |
|---|---|
| SW-dotplot >> window-MMD in the realistic flank-diluted regime | AUC 0.993 vs 0.651 (Exp B) |
| localizes the true core exactly | 140/140 = 100%, median overlap 1.00 (Exp C) |
| cheaper than window-MMD | ~0.9 ms vs 65 ms per pair, compiled (~75x); embed-once keeps GPU ~100x lower |
| representation is a minor lever; keep k16 | SW-dotplot AUC ~0.99 at k16/k32/k64/raw; whole-MMD plateaus ~0.94 by k64 (Exp A) |
| finding PCA-64 is wrong for matching | 0.70 -> keep finding & matching projections separate |
| RNA-FM's 'weak 0.65' was flank-dilution + noisy pipeline pairs, not a model limit | clean Rfam GT: whole-MMD 0.94; SW-dotplot 0.99 |

**Implications for `island_alignment.py`:** swap the RiNALMo scoring (window-grid MMD + SW-over-windows) for dotplot + nt-SW; keep the k16 matching PCA, the executor path, and the assignment/collinearity/output layer. Compile the SW (numba/cython) for the ~70x. No new PCA or executor change. RNA-FM keeps its window-MMD path (separated).

## 5. Deployed-matcher threshold calibration

Pick `max_match_dist` for `model_registry` on the Rfam flank-diluted pos/neg set, using the SHIPPED matcher (`modules/pipeline/matchers/rinalmo`). Quality = `1/(1+SW_score)` (lower=better); the SW score integrates similarity over band length and is the real discriminator (mean-cos alone is weak). Filter also requires `eff_nt >= min_match_eff_nt` (=40).

In [12]:
import importlib, numpy as np
from sklearn.metrics import roc_auc_score
import modules.pipeline.matchers.rinalmo as rm; importlib.reload(rm)
TAU, GAP = 0.5, 0.3   # benchmark-validated (Exp B); also the registry defaults

def score_eff(ref, q):
    s, r0, r1, q0, q1, _ = rm._dotplot_sw(Pj(ref), Pj(q), TAU, GAP)
    return (s, (((r1 - r0) + (q1 - q0)) // 2) if s > 0 else 0)

P = [score_eff(r, q) for r, _, q, _ in planted_pos]
N = [score_eff(r, q) for r, _, q, _ in planted_neg]
ps = np.array([s for s, _ in P]); ns = np.array([s for s, _ in N])
pe = np.array([e for _, e in P]); ne = np.array([e for _, e in N])
lab = np.r_[np.ones(len(ps)), np.zeros(len(ns))]
pd = 1.0 / (1.0 + np.maximum(ps, 0)); nd = 1.0 / (1.0 + np.maximum(ns, 0))
print('AUC by SW score:', round(roc_auc_score(lab, np.r_[ps, ns]), 3))
best = (0, 0.1, 0, 0)
for t in np.linspace(0.02, 0.5, 97):
    tp = ((pd <= t) & (pe >= 40)).mean(); fp = ((nd <= t) & (ne >= 40)).mean()
    if tp - fp > best[0]: best = (tp - fp, t, tp, fp)
print('Youden max_match_dist=%.3f (score>=%.1f): TPR=%.3f FPR=%.3f' % (best[1], 1/best[1]-1, best[2], best[3]))
print('-> registry uses max_match_dist=0.1 (recall-first round; score>=9), min_match_eff_nt=40')

AUC by SW score: 0.993
Youden max_match_dist=0.095 (score>=9.5): TPR=0.979 FPR=0.014
-> registry uses max_match_dist=0.1 (recall-first round; score>=9), min_match_eff_nt=40
